# Notebook 04 — Calibrate the Automated PR Reviewer

Now the moment of truth. We run the automated PR reviewer against each gold-standard entry's diff (using the matching rubric) and measure how often it agrees with your human verdict, dimension by dimension.

**Target**: ≥80% per-dimension agreement. Below that, do not trust the reviewer for scoring downstream — iterate on the rubric instead.

Agreement failure modes:

- **Rubric too loose** — the automated reviewer passes things the human failed. Fix: add red flags or tighten criteria.
- **Rubric too strict** — the automated reviewer fails things the human passed. Fix: soften criteria or reconsider the human verdict.
- **Judgment call dimension** — you and the LLM are both defensible. Fix: either pick a side and encode it, or drop the dimension.

## 1. Run calibration

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '.')

from validators.gold_standard import load_gold_set
from validators.calibration import calibrate

GOLD_DIR = Path('scaffolding/gold_standard')
entries = load_gold_set(GOLD_DIR)
print(f'Running reviewer against {len(entries)} gold entries…')

report = calibrate(entries)
print()
print(report.summary())

## 2. Per-entry breakdown

In [ ]:
import pandas as pd
per_entry = pd.DataFrame([{
    'pr': e.slug,
    'pr_title': e.pr_title,
    'agreement_rate': round(e.agreement_rate, 2),
    'human_overall': e.overall_human,
    'auto_overall': e.overall_auto,
    'overall_agree': e.overall_agree,
} for e in report.entries])
per_entry

## 3. Where do we disagree?

This is the most useful table in the whole workshop. Each row is a specific dimension where the automated reviewer said one thing and you said another. The `auto_reason` column explains why the LLM made its call — that's your lever for tightening the rubric.

In [ ]:
disagreements = report.disagreements()
pd.set_option('display.max_colwidth', 120)
disagreements

## 4. ASK CLAUDE — propose rubric edits

If you have disagreements, send them to Claude and ask for fixes:

In [ ]:
print(f'''Copy-paste this prompt into your Claude Code session:

---
The automated PR reviewer disagreed with me on the following dimensions:

{disagreements.to_string(index=False) if not disagreements.empty else '(no disagreements — skip this step)'}

For each disagreement, decide whether:
  (a) The rubric dimension is underspecified — the LLM interpreted it
      reasonably but not the way I did. Propose a concrete rubric edit
      that would make the LLM reach my verdict, and show it as a diff.
  (b) My verdict was a judgment call and the LLM's read is defensible.
      Suggest I update my gold standard entry instead.

Do NOT apply any edits yet. Show me the proposed changes and I'll
approve them.
---
''')

## 5. Re-run after fixes

After Claude proposes rubric edits and you've applied the ones you agree with, **re-run cell 1** (the calibration cell). Your agreement rate should climb. Iterate until you're ≥80%, or you've decided the remaining disagreements are real judgment calls you're willing to live with.

When you're satisfied, the reviewer is **calibrated for this repo**. You can confidently use it as a signal in the final eval and drop it into CI.

## Next

The reviewer is the rubric judge for the autonomous eval. Before we run anything, the next notebook directs Claude to **build a custom coding agent** so we can compare it to off-the-shelf Claude Code / Kiro.